# 🔴 Day 13 — Agentic RAG (Chat with Your Documents)

**LangGraph from Zero to Advanced · Day 13 of 16**

**Today's goal:** let an LLM answer questions from **your own documents** — and make it smart enough to decide *when* to search, check if results are relevant, and retry with a better query.

| Lesson | Topic |
|---|---|
| 1 | What is RAG? |
| 2 | Build a knowledge base (split → embed → store) |
| 3 | Simple RAG graph: retrieve → answer |
| 4 | Agentic RAG: the retriever as a tool |
| 5 | Self-correcting RAG: grade → rewrite → retry |
| 6 | Practice |

💰 Embeddings use `text-embedding-3-small` — fractions of a paisa for this notebook.

In [ ]:
%pip install -q -U langgraph langchain langchain-openai langchain-text-splitters python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from IPython.display import Image, display

load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

def show_graph(app):
    """Draw the graph as a picture, or print it as text when offline."""
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

print("✅ Ready")

## Lesson 1 — What is RAG?

The LLM doesn't know **your** data (company policies, notes, PDFs). **RAG = Retrieval-Augmented Generation:**

1. **Retrieve** the most relevant chunks of your documents.
2. **Augment** the prompt with them.
3. **Generate** an answer grounded in those chunks.

How does search find "relevant" text? **Embeddings** turn text into lists of numbers where similar meanings are close together. A **vector store** finds the closest chunks to your question.

```
documents ─► split into chunks ─► embed ─► vector store
                                               │
question ──────────────► embed ─► similarity search ─► top chunks ─► LLM ─► answer
```

## Lesson 2 — Build a knowledge base

Our documents: the handbook of a made-up online school, **"Chai & Code Academy"**. (Replace with your own text, or load PDFs with `PyPDFLoader` later.)

In [ ]:
from langchain_core.documents import Document

handbook = {
    "refunds.md": """Refund policy. Students can get a full refund within 7 days of purchase if they
have watched less than 20% of the course. After 7 days, no refunds are given, but the course
can be transferred once to a friend. Refunds are processed to the original payment method within 5 working days.""",
    "certificates.md": """Certificates. A certificate is issued when a student completes all lessons and scores
at least 60% in the final quiz. The quiz can be attempted 3 times. Certificates are emailed as PDF
within 48 hours and can be verified on our website using the certificate ID.""",
    "schedule.md": """Live classes. Live doubt-clearing sessions happen every Saturday at 7 PM IST on Zoom.
Recordings are uploaded within 24 hours. The LangGraph batch starts on the 1st of every month and
runs for 16 days, one lesson per day.""",
    "support.md": """Support. Email help@chaiandcode.example for account issues. The Telegram community is the fastest
way to get coding help. Mentors reply within 12 hours on working days. Office hours are Monday to Friday, 10 AM to 6 PM IST.""",
}
docs = [Document(page_content=text, metadata={"source": name}) for name, text in handbook.items()]
print(len(docs), "documents")

**Split** long documents into small chunks so each search result is focused:

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.split_documents(docs)
print(len(chunks), "chunks")
print("Example chunk:", chunks[0].page_content, "| from", chunks[0].metadata["source"])

**Embed** and **store** them. `InMemoryVectorStore` needs no database — perfect for learning. (For real apps: Chroma, FAISS, pgvector, Pinecone…)

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})     # top-2 chunks

for d in retriever.invoke("Can I get my money back?"):
    print(f"📄 [{d.metadata['source']}] {d.page_content[:90]}...")

## Lesson 3 — Simple RAG graph: retrieve → answer

```
START ► retrieve ► generate ► END
```

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class RAGState(TypedDict):
    question: str
    context: list[Document]
    answer: str

def retrieve(state: RAGState) -> dict:
    return {"context": retriever.invoke(state["question"])}

def generate(state: RAGState) -> dict:
    context = "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in state["context"])
    prompt = ("Answer the question using ONLY the context. If the answer is not in the context, say "
              "\"I don't know\". Mention the source file in brackets.\n\n"
              f"Context:\n{context}\n\nQuestion: {state['question']}")
    return {"answer": llm.invoke(prompt).content}

g = StateGraph(RAGState)
g.add_sequence([retrieve, generate])
g.add_edge(START, "retrieve")
g.add_edge("generate", END)
rag = g.compile()

for q in ["How many times can I attempt the final quiz?", "When are live classes?", "Who won the 2011 World Cup?"]:
    print(f"❓ {q}\n💬 {rag.invoke({'question': q})['answer']}\n")

⚠️ Notice the last question: a good RAG app says **"I don't know"** instead of making things up.

## Lesson 4 — Agentic RAG: the retriever as a tool

Simple RAG **always** searches — even for "hi". An **agent** decides itself: search when needed, maybe search twice, or answer directly. We just wrap the retriever in a `@tool` and reuse the Day 6 agent loop.

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

@tool
def search_handbook(query: str) -> str:
    """Search the Chai & Code Academy handbook (refunds, certificates, live classes, support)."""
    found = retriever.invoke(query)
    return "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in found)

rag_tools = [search_handbook]
agent_llm = llm.bind_tools(rag_tools)
AGENT_SYSTEM = SystemMessage(content="You are the Chai & Code Academy assistant. For questions about the academy, "
                                     "ALWAYS use search_handbook and cite the source file. For small talk, just reply.")

def agent(state: MessagesState) -> dict:
    return {"messages": [agent_llm.invoke([AGENT_SYSTEM] + state["messages"])]}

g = StateGraph(MessagesState)
g.add_node("agent", agent)
g.add_node("tools", ToolNode(rag_tools))
g.add_edge(START, "agent")
g.add_conditional_edges("agent", tools_condition)
g.add_edge("tools", "agent")
rag_agent = g.compile()
show_graph(rag_agent)

In [ ]:
for q in ["Hi there!", "If I bought the course 3 days ago and watched 10%, can I get a refund, and how fast?"]:
    out = rag_agent.invoke({"messages": [("user", q)]})
    searched = any(m.type == "tool" for m in out["messages"])
    print(f"❓ {q}\n🔎 searched: {searched}\n💬 {out['messages'][-1].content}\n")

## Lesson 5 — Self-correcting RAG

Users ask vague questions ("that saturday thing?"). If retrieved chunks are not relevant, **rewrite** the question and try again.

```
START ► retrieve ► grade ─ relevant ──────────────► generate ► END
            ▲         └─ not relevant (max 2) ─► rewrite ─┘
            └─────────────────────────────────────────┘
```

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class Relevance(BaseModel):
    relevant: Literal["yes", "no"] = Field(description="yes if the context helps answer the question")

grader = llm.with_structured_output(Relevance)

class CRAGState(TypedDict):
    question: str
    original_question: str
    context: list[Document]
    relevant: str
    rewrites: int
    answer: str

def c_retrieve(state):
    return {"context": retriever.invoke(state["question"]),
            "original_question": state.get("original_question") or state["question"]}

def c_grade(state):
    ctx = "\n".join(d.page_content for d in state["context"])
    r = grader.invoke(f"Question: {state['original_question']}\nContext:\n{ctx}")
    print(f"🧪 grade for '{state['question']}': {r.relevant}")
    return {"relevant": r.relevant}

def c_route(state) -> Literal["generate", "rewrite"]:
    if state["relevant"] == "yes" or state.get("rewrites", 0) >= 2:
        return "generate"
    return "rewrite"

def c_rewrite(state):
    better = llm.invoke("Rewrite this question as a clear search query for an online-course handbook "
                        f"(refunds, certificates, live classes, support). Reply with the query only: {state['original_question']}").content
    print("✏️ rewritten to:", better)
    return {"question": better, "rewrites": state.get("rewrites", 0) + 1}

def c_generate(state):
    return generate({"question": state["original_question"], "context": state["context"]})   # reuse Lesson 3

g = StateGraph(CRAGState)
g.add_node("retrieve", c_retrieve)
g.add_node("grade", c_grade)
g.add_node("rewrite", c_rewrite)
g.add_node("generate", c_generate)
g.add_edge(START, "retrieve")
g.add_edge("retrieve", "grade")
g.add_conditional_edges("grade", c_route)
g.add_edge("rewrite", "retrieve")
g.add_edge("generate", END)
crag = g.compile()
show_graph(crag)

In [ ]:
out = crag.invoke({"question": "that weekend zoom thing, what time?", "rewrites": 0})
print("\n💬", out["answer"])

## Lesson 6 — Practice

**Exercise 1.** Add a new document `fees.md` ("The LangGraph course costs ₹2,999. Students get 20% off with code CHAI20.") to the vector store with `vector_store.add_documents(...)`, then ask the simple `rag` graph about the price.

**Exercise 2.** Change the simple RAG graph so `generate` also returns `sources: list[str]` (unique file names used).

In [ ]:
# ✅ Solution 1
vector_store.add_documents([Document(page_content="Fees. The LangGraph course costs ₹2,999. Students get 20% off with code CHAI20.",
                                     metadata={"source": "fees.md"})])
print(rag.invoke({"question": "How much does the LangGraph course cost and is there a discount?"})["answer"])

In [ ]:
# ✅ Solution 2
class RAGWithSources(RAGState):
    sources: list[str]

def generate_with_sources(state):
    out = generate(state)
    out["sources"] = sorted({d.metadata["source"] for d in state["context"]})
    return out

g = StateGraph(RAGWithSources)
g.add_node("retrieve", retrieve)
g.add_node("generate", generate_with_sources)
g.add_edge(START, "retrieve")
g.add_edge("retrieve", "generate")
g.add_edge("generate", END)
res = g.compile().invoke({"question": "How do I verify a certificate?"})
print(res["answer"], "\n📚", res["sources"])
assert isinstance(res["sources"], list)
print("✅ All exercises solved!")

## 🎯 Day 13 Recap

* RAG = retrieve relevant chunks → put them in the prompt → generate.
* Pipeline: split → embed → vector store → retriever.
* **Agentic RAG:** retriever as a tool; the agent decides when to search.
* **Self-correcting RAG:** grade relevance → rewrite query → retry (with a limit).
* Always tell the model to say "I don't know" and cite sources.

**Tomorrow (Day 14):** **long-term memory** across conversations with the `Store`, and runtime **context**.